# Clasificación de sesiones de *vishing* con TabFM (backend JAX + GPU)

Adapta `examples/vishing_classification.py` a un notebook pensado para correr en
**SageMaker** con **JAX sobre GPU**. Predice `is_vishing` a partir de señales de
comportamiento del dataset sintético de BioCatch (`data/biocatch_sinthetic_data.csv`).

**Decisiones de diseño:**
- **Modelo de señal propia:** se excluyen las columnas del proveedor (`biocatch_*`).
- **Sin data leakage:** se excluyen identificadores y metadata posterior al reclamo
  (`days_to_claim`, `claim_category`).
- **In-context learning:** TabFM no entrena pesos; usa el *train* como contexto en
  inferencia. El costo crece con el nº de filas de contexto → se usa un contexto
  estratificado (con GPU se puede subir `CONTEXT_SIZE`).
- **Desbalance ~5%:** se reporta PR-AUC, ROC-AUC, matriz de confusión y barrido de umbrales.

> **Importante:** la celda 0 configura el asignador de memoria de GPU y **debe ejecutarse
> antes de cualquier `import jax`**. Si ya importaste jax en este kernel, haz
> **Kernel → Restart** y ejecuta desde arriba.

## 0. Configurar memoria de GPU y verificar entorno

Por defecto JAX **preasigna el 75% de la VRAM** en un pool (BFC). Al restaurar el
checkpoint de TabFM (orbax deserializa muchos arrays en paralelo) eso fragmenta la
memoria y provoca `RESOURCE_EXHAUSTED` aunque la GPU tenga espacio. Usamos el
asignador `platform` (reserva bajo demanda y libera), que evita el problema.

In [ ]:
import os

# --- DEBE ir antes del primer 'import jax' ---
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

import jax
print("JAX:", jax.__version__)
print("Devices:", jax.devices())
# Debe listar al menos un dispositivo CUDA/Gpu. Si solo aparece CpuDevice, revisa
# la instalacion de jax[cuda12] y que la instancia tenga GPU (ejecuta !nvidia-smi).

In [ ]:
!nvidia-smi

## 1. Imports y configuración

In [ ]:
import time

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

import tabfm

In [ ]:
# Ruta al CSV (relativa a examples/). Ajusta si subiste el repo a otra ubicacion.
DATA_PATH = os.path.join("..", "data", "biocatch_sinthetic_data.csv")

TARGET = "is_vishing"

ID_COLS = ["session_id", "customer_id", "session_timestamp"]
LEAKAGE_COLS = ["days_to_claim", "claim_category"]  # metadata posterior al reclamo
BIOCATCH_COLS = [
    "biocatch_risk_score", "biocatch_genuine_score", "biocatch_ato_indicator",
    "biocatch_social_eng_indicator", "biocatch_bot_indicator",
]

# Con GPU + buena RAM puedes subir el contexto. En T4 (16 GB) 5-8k es comodo;
# en A10G (24 GB) puedes probar 8-12k. Mas contexto = mas señal pero mas lento.
CONTEXT_SIZE = 5000
TEST_SIZE = 3000
N_ESTIMATORS = 16   # miembros del ensemble; en GPU puedes subir a 32.
RANDOM_STATE = 42

## 2. Carga del dataset (features de señal propia)

In [ ]:
def load_dataset(path=DATA_PATH):
    df = pd.read_csv(path)
    drop_cols = ID_COLS + LEAKAGE_COLS + BIOCATCH_COLS + [TARGET]
    feature_cols = [c for c in df.columns if c not in drop_cols]
    X = df[feature_cols].copy()
    y = df[TARGET].astype(int)
    return X, y

X, y = load_dataset()
print(f"Sesiones: {len(X)} | Features: {X.shape[1]} | Prevalencia vishing: {y.mean():.4f}")
print("Features usadas:", list(X.columns))
X.head()

## 3. Contexto (in-context) y hold-out estratificados

In [ ]:
def stratified_sample(X, y, n):
    """Muestra estratificada de tamaño n (o todo X si n >= len(X))."""
    if n >= len(X):
        return X, y
    Xs, _, ys, _ = train_test_split(
        X, y, train_size=n, stratify=y, random_state=RANDOM_STATE)
    return Xs, ys

X_pool, X_test, y_pool, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
X_ctx, y_ctx = stratified_sample(X_pool, y_pool, CONTEXT_SIZE)

print(f"Contexto: {len(X_ctx)} filas ({int(y_ctx.sum())} vishing) | "
      f"Test: {len(X_test)} filas ({int(y_test.sum())} vishing)")

## 4. Cargar modelo TabFM (backend JAX / GPU)

In [ ]:
t0 = time.time()
model = tabfm.tabfm_v1_0_0_jax.load(model_type="classification")
print(f"Modelo cargado en {time.time() - t0:.1f}s")

clf = tabfm.TabFMClassifier(
    model=model, n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE)

## 5. Fit (fija el contexto) y predicción

La primera llamada compila (XLA) y puede tardar; las siguientes son rápidas.

In [ ]:
t0 = time.time()
clf.fit(X_ctx, y_ctx)
print(f"fit() en {time.time() - t0:.1f}s")

t0 = time.time()
proba = clf.predict_proba(X_test)
print(f"predict_proba() en {time.time() - t0:.1f}s")

# Probabilidad de la clase positiva (vishing == 1).
vishing_idx = list(clf.classes_).index(1)
proba_vishing = proba[:, vishing_idx]

## 6. Evaluación (métricas para clase desbalanceada)

In [ ]:
def evaluate(y_true, proba_vishing):
    roc = roc_auc_score(y_true, proba_vishing)
    pr_auc = average_precision_score(y_true, proba_vishing)
    print("=== Metricas globales (independientes del umbral) ===")
    print(f"ROC-AUC          : {roc:.4f}")
    print(f"PR-AUC (avg prec): {pr_auc:.4f}  (baseline = prevalencia = {y_true.mean():.4f})")
    print("\n=== Barrido de umbrales (positivo = vishing) ===")
    print(f"{'umbral':>7} | {'precision':>9} | {'recall':>7} | {'f1':>6} | {'flagged':>7}")
    for thr in (0.3, 0.5, 0.7):
        y_pred = (proba_vishing >= thr).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="binary", zero_division=0)
        print(f"{thr:>7.2f} | {p:>9.3f} | {r:>7.3f} | {f1:>6.3f} | {int(y_pred.sum()):>7}")
    print("\n=== Reporte de clasificacion (umbral 0.5) ===")
    y_pred_05 = (proba_vishing >= 0.5).astype(int)
    print(classification_report(
        y_true, y_pred_05, target_names=["legitima", "vishing"], digits=3,
        zero_division=0))
    print("Matriz de confusion (filas=real, cols=pred) [legitima, vishing]:")
    print(confusion_matrix(y_true, y_pred_05))

evaluate(y_test.to_numpy(), proba_vishing)